In [8]:
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image, ImageDraw, ImageFilter
import numpy as np

# 1. KONFIGURACJA URZĄDZENIA (Apple Silicon M4)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Używane urządzenie: {DEVICE}")

# 2. ŁADOWANIE POTOKU (Pipeline)
# Używamy modelu dedykowanego do inpaintingu (9-kanałowego)
model_id = "runwayml/stable-diffusion-inpainting"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # float16 drastycznie przyspiesza generowanie na Macu
    use_safetensors=True,
    variant="fp16"
)
pipe.to(DEVICE)

# Opcjonalne: Przyspieszenie (Memory Efficient Attention)
# pipe.enable_attention_slicing()

# 3. PRZYGOTOWANIE OBRAZÓW WEJŚCIOWYCH
# Wczytujemy zdjęcie psa
image_path = "../images/dog.jpg"
try:
    init_image = Image.open(image_path).convert("RGB").resize((512, 512))
except FileNotFoundError:
    # Generator zastępczy, jeśli nie masz pliku
    init_image = Image.new("RGB", (512, 512), (100, 100, 100))

# Tworzymy maskę (biały = obszar do zmiany, czarny = zachowany)
mask_image = Image.new("L", (512, 512), 0)
draw = ImageDraw.Draw(mask_image)

# Rysujemy prostokąt tam, gdzie ma być kapelusz
# Twoje współrzędne: [x0, y0, x1, y1]
draw.rectangle([150, 50, 360, 250], fill=255)

# Rozmywamy krawędzie maski, aby przejście było naturalne
mask_image = mask_image.filter(ImageFilter.GaussianBlur(radius=15))

# 4. PARAMETRY GENEROWANIA
prompt = "A close-up of a dog wearing stylish black hat, perfectly fitted to the face, realistic textures, sharp focus, cinematic lighting, 8k, highly detailed fur, professional photography"
negative_prompt = "blurry, bad anatomy, distorted, deformed, ugly, low quality"

# 5. URUCHOMIENIE GENEROWANIA
# Diffusers automatycznie zajmuje się szumem, timestepami i łączeniem kanałów
generator = torch.Generator(DEVICE).manual_seed(44) # Dla powtarzalności wyników

output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    mask_image=mask_image,
    guidance_scale=9.0,      # Twoje CFG Scale
    num_inference_steps=50,  # Liczba kroków
    strength=0.95,           # Jak bardzo AI może odejść od oryginału wewnątrz maski
    generator=generator
).images[0]

# 6. ZAPIS I WYŚWIETLENIE
output.save("dog_with_hat.png")
print("Sukces! Obraz został zapisany jako dog_with_hat.png")

# Jeśli pracujesz w Jupyter Notebook / Google Colab:
output.show()

Używane urządzenie: mps


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 7701.67it/s]

100%|██████████| 47/47 [01:04<00:00,  1.37s/it]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Sukces! Obraz został zapisany jako dog_with_hat.png
